# How does a wall change a vortex collision?

A red, counterclockwise vortex starts to the left of a blue, clockwise vortex. Each induces upward velocity at the center of the other. Together they form a *dipole* that travels toward the upper wall. Since fluid cannot cross the wall, the incoming flow must turn. Our question is how the condition on **motion along the wall** changes that turn and the energy lost during it.

We use the same two-dimensional incompressible Navier–Stokes solver, initial velocity, and viscosity for both runs. We follow the cores before and after their closest approach, then measure how much of the initial kinetic energy viscosity removes in the top sixteenth of the box. The working prediction is that a no-slip wall will produce a thin shear layer at impact. Its induced velocity should turn the cores back toward the interior, and its large velocity gradient should produce an early burst of dissipation.

Run the cells in order to inspect the signed shear at impact, watch the collision, and reproduce the two-grid energy comparison.

## The controlled experiment

Take $x$ to the right and $y$ upward. The upper wall is at $y=1$. Write $u$ for horizontal velocity and $v$ for vertical velocity. All walls are stationary and impermeable. Only the upper wall changes between runs.

| Upper wall | Conditions | What the fluid can do |
|---|---|---|
| Free slip | $v=0$, $\partial_y u=0$ | Move along the wall with zero tangential stress. |
| No slip | $v=0$, $u=0$ | Match the wall's zero velocity in both directions. |

The other three walls use free slip. The square has unit side length and the kinematic viscosity is 0.001. The initial streamfunction vanishes near every boundary, so both wall conditions admit exactly the same starting flow. The first cell checks this on the grid used for the movie.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, Video, display
from scipy.integrate import trapezoid

from simulations.run_2d_wall_collision import matched_pair, sampled_runs
from simulations.run_2d_wall_dissipation import compare

free, rigid = matched_pair(193)
np.testing.assert_allclose(free.velocity(), rigid.velocity(), atol=1e-12)
np.testing.assert_allclose(free.vorticity(), rigid.vorticity(), atol=1e-12)
for case in (free, rigid):
    print(
        f"{case.walls['top'].kind:9s}  initial kinetic energy = {case.kinetic_energy():.6f}"
    )

## Two checks on the solver

**Free-slip decay.** A sinusoidal vorticity field has an exact solution in which viscosity reduces the amplitude while preserving the shape. On the right, halving the grid spacing reduces the final field error by a factor of about four. This tests spatial accuracy for a smooth solution.

![Exact mode decay and spatial convergence](figures/walls/free_slip_decay.png)

**Moving no-slip lid.** A lid moving to the right sets the interior fluid in motion. At Reynolds number 100, the two centerline profiles approach the published numerical reference values as the grid is refined. The maximum discrepancy on the finer grid is 0.59% of lid speed. The right-hand panel shows when the computed velocity field settles. [Ghia, Ghia, and Shin, Tables I and II](https://doi.org/10.1016/0021-9991%2882%2990058-4)

![Cavity centerline profiles and approach to steady flow](figures/walls/cavity_profiles.png)

## Why a blue layer appears above the red core

Above the left, counterclockwise core, the fluid moves **left**, so $u<0$ just below the upper wall. With no slip, $u$ must rise from that negative value to zero as $y$ increases toward the wall. Thus $\partial_y u>0$ there. The vertical velocity vanishes everywhere along the wall, giving $\partial_x v=0$ on its flat face. The signed vorticity at the wall is therefore
$$
\omega=\partial_x v-\partial_y u=-\partial_y u<0.
$$
That gives a clockwise (blue) layer above the counterclockwise (red) core. Above the right core the outer flow points right, the gradient changes sign, and the wall layer is red. A free-slip wall allows the horizontal velocity to persist all the way to the boundary. Its zero tangential stress gives zero vorticity on the flat wall itself.

The shear argument gives the sign of the vorticity. Its **supply from the wall** follows from tangential momentum balance. At a stationary no-slip wall the tangential acceleration is zero, so the pressure gradient induced by the approaching dipole balances viscous diffusion. With $y$ pointing up, the balance is $\partial_x p/\rho=-\nu\partial_y\omega$, where $p$ is pressure and $\rho$ is density. Thus the tangential pressure gradient sets the diffusive vorticity flux at the wall. [Wall vorticity balance, equation (11)](https://doi.org/10.1063/1.2814345)

The profiles below cut vertically through the left side of the collision at $x\approx0.43$, $t=0.55$. Follow each curve upward. The no-slip velocity turns sharply toward zero over the last few grid points, exactly where the vorticity becomes strongly negative. The dots are computed grid values. [View the computed profiles](figures/walls/impact_shear_profiles.png)

In [ ]:
for t, free_case, rigid_case in sampled_runs(129):
    if t >= 0.55:
        break

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5), sharey=True, layout="constrained")
for case, label, color in (
    (free_case, "Free slip", "#126987"),
    (rigid_case, "No slip", "#b04729"),
):
    i = np.abs(case.X[:, 0] - 0.43).argmin()
    upper = case.Y[i] >= 0.84
    y = case.Y[i, upper]
    u, _ = case.velocity()
    for ax, field in zip(axes, (u, case.vorticity())):
        ax.plot(field[i, upper], y, ".-", color=color, label=label)
for ax, xlabel in zip(axes, ("Horizontal velocity", "Signed vorticity")):
    ax.axvline(0, color="0.7", linewidth=0.8, zorder=0)
    ax.axhline(1, color="0.3", linewidth=0.8)
    ax.set(xlabel=xlabel, ylim=(0.84, 1.01))
axes[0].set_ylabel("Height y (wall at 1)")
axes[0].legend()
fig.suptitle(f"Above the left core at x = {case.X[i, 0]:.3f}, t = {t:.2f}")
plt.show()

## How the wall layer turns the cores

With free slip, the flow turns along the wall and the cores separate sideways. With no slip, viscosity transmits the boundary's influence into the fluid. The incoming flow stretches and sweeps the new layer along the wall. Where part of the layer lifts off and curls around a primary vortex, it also induces velocity at that vortex. For example, a patch of **clockwise** vorticity on the **left flank** of the red core induces a **downward** velocity at the core. The mirrored arrangement on the right gives a downward component there too. The cores can now travel obliquely back into the box. This interaction between a primary vortex and wall-generated vorticity is the mechanism behind viscous rebound in dipole–wall collisions. [Kramer, Clercx, and van Heijst (2007), sections IV–V](https://doi.org/10.1063/1.2814345)

Pause near simulation time **0.55** to find the oppositely signed layers above the cores. Advance to **0.90** and compare their distances from the wall. The movie shows the curling layers and the resulting trajectories. Quantifying the share of the induced velocity due to the secondary vorticity would require a separate velocity decomposition. Both panels use the same clock and color scale, which saturates at ±75. [Open the movie](figures/walls/vortex_wall_collision.mp4)

In [ ]:
movie_path = Path("figures/walls/vortex_wall_collision.mp4")
Video(filename=str(movie_path), embed=True, width=850)

## When do the core paths separate?

We track each primary core by its original vorticity sign. At each saved time we locate the strongest same-sign interior extremum within 0.04 box lengths of its previous position. This local search follows the coherent rotation center as it moves. A core center can migrate relative to individual fluid parcels. We exclude the wall nodes, where the no-slip layer can have stronger vorticity than the core.

The left panel shows both paths on the fine grid. The right panel measures the perpendicular distance from the upper wall, averaged over the two mirror cores. We define rebound as the largest increase in that distance after closest approach through time 1.20. To mark a reproducible divergence time, we use the first three consecutive snapshots with mean cross-case core separation of at least 0.02 box lengths. That threshold describes a visible separation, not the first physical influence of the wall. [View the trajectory figure](figures/walls/vortex_core_paths.png)

In [ ]:
def core_paths(n):
    h = 1 / (n - 1)
    xy = np.array([[[0.45, 0.52], [0.55, 0.52]]] * 2)
    history, times = [], []
    for t, free_case, rigid_case in sampled_runs(n):
        for case_index, case in enumerate((free_case, rigid_case)):
            omega = case.vorticity()
            for core, sign in enumerate((1, -1)):
                x, y = xy[case_index, core]
                inside = (case.X - x) ** 2 + (case.Y - y) ** 2 <= 0.04**2
                inside[[0, -1], :] = False
                inside[:, [0, -1]] = False
                i, j = np.unravel_index(
                    np.argmax(np.where(inside, sign * omega, -np.inf)), omega.shape
                )
                if (
                    sign * omega[i, j] <= 0
                    or not inside[i - 1 : i + 2, j - 1 : j + 2].all()
                ):
                    raise ValueError("Core peak lost or reached the search boundary")
                # Parabolic offsets remove most of the node-to-node stepping.
                offsets = []
                for a, c in (
                    (omega[i - 1, j], omega[i + 1, j]),
                    (omega[i, j - 1], omega[i, j + 1]),
                ):
                    curvature = a - 2 * omega[i, j] + c
                    offsets.append(0.5 * (a - c) / curvature if curvature else 0.0)
                xy[case_index, core] = (np.array([i, j]) + offsets) * h
        times.append(t)
        history.append(xy.copy())
    return np.array(times), np.array(history)


fig, (paths, gaps) = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
for n in (129, 193):
    t, xy = core_paths(n)
    distance = 1 - xy[..., 1].mean(axis=2)
    separation = np.linalg.norm(xy[:, 0] - xy[:, 1], axis=-1).mean(axis=1)
    above = separation >= 0.02
    first = np.flatnonzero(above[:-2] & above[1:-1] & above[2:])[0]
    closest = distance[:, 1].argmin()
    farthest = closest + distance[closest:, 1].argmax()
    rebound = distance[farthest, 1] - distance[closest, 1]
    print(
        f"{n}²: separation threshold at t={t[first]:.2f}; no-slip gap "
        f"{distance[closest, 1]:.3f} L at t={t[closest]:.2f} to "
        f"{distance[farthest, 1]:.3f} L at t={t[farthest]:.2f}; rebound {rebound:.3f} L"
    )
    for case, (name, color) in enumerate(
        (("Free slip", "#126987"), ("No slip", "#b04729"))
    ):
        gaps.plot(
            t,
            distance[:, case],
            color=color,
            linestyle="-" if n == 193 else ":",
            label=f"{name}, {n}²",
        )
        if n == 193:
            for core in (0, 1):
                paths.plot(
                    *xy[:, case, core].T, color=color, label=name if core == 0 else None
                )
            paths.scatter(*xy[first, case].T, color=color, marker="x")
    if n == 193:
        paths.scatter(*xy[0, 0].T, marker="o", color="black", s=18, label="Start")
        paths.scatter([], [], marker="x", color="black", label="Separation threshold")
        gaps.scatter(
            t[[closest, farthest]],
            distance[[closest, farthest], 1],
            color="#b04729",
            zorder=4,
        )
paths.axhline(1, color="black")
paths.set(
    xlim=(0, 1),
    ylim=(0.48, 1.02),
    aspect="equal",
    xlabel="x / L",
    ylabel="y / L",
    title="Primary core trajectories",
)
paths.legend(fontsize=8)
gaps.set(
    xlabel="Time", ylabel="Core distance from wall / L", title="Approach and rebound"
)
gaps.legend(fontsize=8)
plt.show()

On both grids, corresponding free-slip and no-slip cores first meet the sustained 2%-of-box-width separation criterion at time **0.62**, just after the no-slip cores reach their closest approach. On the 193² grid, their mean distance from the upper wall rises from **0.075 box widths at time 0.59** to **0.129 at time 1.20**. The rebound measured from the unrounded positions is **0.055 box widths**, compared with **0.054** on the 129² grid. The farthest post-impact point occurs in the last frame, so this is the rebound observed within the movie.

## Why the collision dissipates energy near the wall

A stationary wall does no mechanical work in either run. Viscosity converts kinetic energy into internal energy in the fluid. The local loss rate per unit mass is
$$
\epsilon=2\nu\sum_{i,j} S_{ij}^2,\qquad S_{ij}=\tfrac12(\partial_i u_j+\partial_j u_i).
$$
The distinction between rotation and strain matters here. A patch in rigid rotation can have large vorticity and zero strain. In the thin no-slip layer, however, tangential velocity changes by an amount $U$ across a thickness $\delta$. If this shear dominates, $\omega\approx-\partial_y u$ and $\epsilon\approx\nu(\partial_y u)^2\sim\nu(U/\delta)^2$. The **same shear** explains the signed layer in the movie and the large positive loss rate near impact. Opposite signs on the two halves both contribute to dissipation.

We integrate the full strain-based density across the top sixteenth of the box, using the same physical strip on both grids and sampling every 0.01 time units. Free-slip flow also dissipates energy as the cores deform and diffuse near the wall. The calculation measures dissipation *inside* the strip. Fluid can carry energy across its lower edge.

![Near-wall dissipation on both grids](figures/walls/vortex_wall_dissipation.png)

The left panel shows the early no-slip burst, peaking at about 2.2 times the free-slip peak on the fine grid. The middle panel gives the strip's share of whole-box dissipation at each instant. After about time 0.78 the free-slip rate is higher, consistent with the cores staying near the wall longer. The right panel integrates each rate over time. The earlier no-slip burst still gives the larger cumulative loss at time 1.20.

### Recompute the four runs

The next cell advances both wall conditions on each grid using the same time stepping as the movie. The last column checks the whole-box energy balance by adding cumulative viscous loss to final kinetic energy and subtracting initial kinetic energy. Neither stationary wall supplies mechanical power, so the continuum residual is zero.

In [ ]:
rows = [
    "| Nodes per side | Upper wall | Peak strip rate | Strip loss / initial energy | Strip loss / total loss | Signed budget residual / initial energy |",
    "|---:|:---|---:|---:|---:|---:|",
]
for nodes in (129, 193):
    times, rates, energies = compare(nodes)
    for index, wall in enumerate(("Free slip", "No slip")):
        total_rate, strip_rate = rates[:, index, :].T
        total_loss = trapezoid(total_rate, x=times)
        strip_loss = trapezoid(strip_rate, x=times)
        initial = energies[0, index]
        residual = (energies[-1, index] + total_loss - initial) / initial
        rows.append(
            f"| {nodes} | {wall} | {strip_rate.max():.5f} | {strip_loss / initial:.2%} "
            f"| {strip_loss / total_loss:.2%} | {residual:+.2%} |"
        )
display(Markdown("\n".join(rows)))

On the fine grid the strip dissipates **17.46% of the initial energy with no slip** and **13.20% with free slip**. The coarse-grid values are 17.30% and 13.18%. The wall effect is larger than the change between these two grids. The fine-grid whole-box budget still has a residual of about 0.51% of initial energy in the no-slip case. These measurements describe this dipole, viscosity, strip width, and time window. [Experiment setup](simulations/run_2d_wall_collision.py) · [Dissipation calculation](simulations/run_2d_wall_dissipation.py)